In [1]:
from wequant.data_processing import PortfolioManager

# objects inported in wequant.data_processing
import os, calendar
import polars as pl
import pandas as pd
from typing import Union, Literal
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
from plotly.graph_objects import Figure

In [2]:
PM = PortfolioManager()

In [7]:
#def get_individual_stocks_info(
#self,
#specific_date: date = date.today(),
#output_performance: bool = False
#) -> pl.DataFrame:　
'''
specific_dateで指定した最新portfolioに含まれる各個別株のファンダメンタルズや
最新決算における業績成長率などのデータを銘柄ごとにpl.DataFrameにまとめて返す。
output_porformance = Trueにすると、 '買値','株価','数量', '保有高', '損益', '口座'の各列を出力する。
'''

# import from module
from wequant.data_processing import FinancequotePl, KessanPl

# para
self = PM
specific_date = date.today()
output_performance = False

In [8]:
# test
df = PM.get_individual_stocks_info(specific_date, output_performance = output_performance)
pl.Config.set_tbl_rows(df.shape[0])
df

date,code,name,fq-PER,fq-配当率,q-sett,q-sgr,q-op,q-pgr
date,str,str,f64,f64,date,f64,i64,f64
2026-02-11,"""1301""","""極洋""",9.22,2.88,2025-09-30,10.51,1717,-15.0
2026-02-11,"""1414""","""ショーボンド HD""",20.44,2.97,2025-09-30,-4.71,4918,-0.22
2026-02-11,"""1721""","""コムシス HD""",20.07,2.25,2025-09-30,2.18,10531,11.23
2026-02-11,"""2325""","""NJS""",20.42,2.08,2025-09-30,7.93,-941,66.25
2026-02-11,"""2393""","""日本ケアサプライ""",17.5,2.91,2025-09-30,7.55,872,34.57
2026-02-11,"""2708""","""久世""",8.19,1.9,2025-09-30,11.87,527,24.59
2026-02-11,"""285A""","""キオクシアＨＤ""",null,null,null,null,null,null
2026-02-11,"""2914""","""日本たばこ産業""",19.1,3.87,2025-09-30,9.24,269341,48.64
2026-02-11,"""3391""","""ツルハ HD""",15.88,1.93,2025-11-30,4.73,11915,1.43


In [49]:
debug_df.columns

['date', 'ticker_code', '銘柄名', 'purchase_price', 'close_price', 'quantity']

In [67]:
# body PM.get_individual_stocks_info
# base
if output_performance:
    cols = [
        "date", 
        "ticker_code",
        "銘柄名",
        'purchase_price',
        'close_price',
        'quantity'
    ]
    new_cols = [
        "date", 
        "code",
        "銘柄名",
        'purchase_price',
        'close_price',
        'quantity',
    ]
else:
    cols = ["date", "ticker_code", "銘柄名"]
    new_cols = ["date", "code", "銘柄名"]

df = self.get_individual_stocks(specific_date, cols)

# debug
# get_individual_stocksメソッドはレコードの重複排除しかしないので、同一銘柄のperformanceをcodeで集約する。
df = df.with_columns([
    pl.col('purchase_price').cast(pl.Float64).alias('purchase_price'),
    pl.col('close_price').cast(pl.Float64).alias('close_price'),
    pl.col('quantity').cast(pl.Int64).alias('quantity')
])

df = df.group_by(["ticker_code"]).agg([
    pl.col('date').last(), 
    pl.col('銘柄名').last(), 
    pl.col('purchase_price').mean().round(1), 
    pl.col('close_price').mean().round(1), 
    pl.col('quantity').sum()
])

# debug
debug_df = df

# key列となるからcodeに変更
df = df.with_columns([
    pl.col("ticker_code").alias("code")
]).select(new_cols)
# 後で使うのでオリジナルとして取得しておく
original_df = df

# finance_quateのデータ
# code, 'expected_PER', expected_dividend_yield
FQ = FinancequotePl()
cols = ['code', 'expected_PER', 'expected_dividend_yield']
df1 = FQ.filter_finance_quotes_by_date(specific_date)
df1 = df1[cols]
# code列を文字列に変更
df1 = df1.with_columns(
    pl.col("code").cast(pl.Utf8)
)
df = df.join(df1, on=["code"], how="left")
# 列名変更
df = df.with_columns([
    pl.col("銘柄名").alias("name"),
    pl.col("expected_PER").alias("fq-PER"),
    pl.col("expected_dividend_yield").alias("fq-配当率")
]).select(["date","code","name","fq-PER","fq-配当率"])

# kessanデータ
# 四半期対前年同期比売上高成長率(q-sgr)と四半期経常利益(q-op)と経常利益成長率列(q-pgr)を追加する
# dfをcode変換してK.dfのレコードを保有銘柄だけに絞れるようにする
holdings = []
for c in df["code"]:
    #print(c)
    try:
        holdings.append(int(c))
    except:
        continue
K = KessanPl()
K.filter_by_codes(holdings)
K.filter_by_settlement_type("四")
K.with_columns_growth_rate()
# specific_dateにおける最新四半期決算のみを抽出
df1 = K.df
df1 = df1.group_by(["code"]).agg([
    pl.col("settlement_date").last(),
    pl.col("gr_sales").last(),
    pl.col("ordinary_profit").last(),
    pl.col("gr_ordinary_profit").last()
])
# codeの型を変換して列名を変更
df1 = df1.with_columns([
    pl.col("code").cast(pl.Utf8).alias("code"),
    pl.col("settlement_date").alias("q-sett"),
    pl.col("gr_sales").alias("q-sgr"),
    pl.col("ordinary_profit").alias("q-op"),
    pl.col("gr_ordinary_profit").alias("q-pgr"),
]).select(["code", "q-sett", "q-sgr", "q-op", "q-pgr"])
# join to df
df = df.join(df1, on=["code"], how="left")

# performance情報を追加する
if not output_performance:
    #return df
    pass

df2 = original_df
# 列の型を見やすく変更する
df2 = df2.with_columns([
    pl.col("purchase_price").cast(pl.Float64),
    pl.col("close_price").cast(pl.Float64),
    pl.col("quantity").cast(pl.Int64)
])

# 損益列を追加し、出力用に列名を変更する
# '買値','現在値','数量', '保有高', '損益', '口座'
df2 = df2.with_columns([
    pl.col("purchase_price").alias("買値"),
    pl.col("close_price").alias("現在値"),
    pl.col("quantity").alias("数量"),
    (pl.col("close_price") * pl.col("quantity")).cast(pl.Int64).alias('保有高'),
    ((pl.col("close_price")-pl.col("purchase_price"))* pl.col("quantity")).cast(pl.Int64).alias('損益')
]).select([
    pl.col("code"),
    pl.col('買値'),
    pl.col('現在値'),
    pl.col('数量'), 
    pl.col('保有高'), 
    pl.col('損益')
])
# join
df = df.join(df2, on=["code"], how="left")
df = df.sort("code")


In [69]:
df

date,code,name,fq-PER,fq-配当率,q-sett,q-sgr,q-op,q-pgr,買値,現在値,数量,保有高,損益
date,str,str,f64,f64,date,f64,i64,f64,f64,f64,i64,i64,i64
2026-02-11,"""1301""","""極洋""",9.22,2.88,2025-09-30,10.51,1717,-15.0,5180.0,5200.0,10,52000,200
2026-02-11,"""1414""","""ショーボンド HD""",20.44,2.97,2025-09-30,-4.71,4918,-0.22,1298.0,1531.0,40,61240,9320
2026-02-11,"""1721""","""コムシス HD""",20.07,2.25,2025-09-30,2.18,10531,11.23,3422.0,5329.0,15,79935,28605
2026-02-11,"""2325""","""NJS""",20.42,2.08,2025-09-30,7.93,-941,66.25,5250.0,5040.0,10,50400,-2100
2026-02-11,"""2393""","""日本ケアサプライ""",17.5,2.91,2025-09-30,7.55,872,34.57,1436.2,2477.0,100,247700,104080
…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-02-11,"""8473""","""SBI HD""",null,null,null,null,null,null,1932.0,3605.0,52,187460,86996
2026-02-11,"""8591""","""オリックス""",13.85,2.21,2025-09-30,14.43,236002,72.49,2574.8,5429.0,100,542900,285420
2026-02-11,"""9433""","""KDDI""",13.44,3.1,2025-09-30,4.11,315779,14.19,2464.0,2580.0,21,54180,2436


In [9]:
df2.columns


['code', '買値', '現在値', '数量', '保有高', '損益', '口座']

In [5]:
K.df

code,settlement_date,settlement_type,announcement_date,sales,operating_income,ordinary_profit,final_profit,reviced_eps,dividend,quater
i64,date,str,date,i64,i64,i64,i64,f64,f64,i64
1301,2017-03-31,"""本""",2017-05-11,236561,3723,3709,2422,230.7,60.0,-2
1301,2018-03-31,"""本""",2018-05-10,254783,4066,4437,3211,304.3,60.0,4
1301,2018-09-30,"""四""",2018-11-05,61245,507,595,269,24.9,0.8,2
1301,2018-12-31,"""四""",2019-02-08,78581,2208,2591,1677,155.2,2.8,3
1301,2019-03-31,"""四""",2019-05-13,58368,551,511,413,38.2,0.9,4
…,…,…,…,…,…,…,…,…,…,…
9757,2024-12-31,"""四""",2025-02-07,8392,2379,2443,1950,41.7,28.3,4
9757,2024-12-31,"""本""",2025-02-07,30645,8324,8411,5993,128.0,75.0,4
9757,2025-03-31,"""四""",2025-05-09,7775,2307,2321,79,1.7,29.7,1


In [7]:
df["code"].unique().to_list()

['9433',
 '3663',
 '6254',
 '3697',
 '6370',
 '3984',
 '8001',
 '6490',
 '6203',
 '2708',
 '5110',
 '1301',
 '1721',
 '4413',
 '285A',
 '4617',
 '3392',
 '8316',
 '6946',
 '3565',
 '3391',
 '2325',
 '5020',
 '8591',
 '6367',
 '5401',
 '4680',
 '7826',
 '8002',
 '2914',
 '6584',
 '4168',
 '5262',
 '4980',
 '6351',
 '8306',
 '6349',
 '6508',
 '4021',
 '7721',
 '4395',
 '9434',
 '2393',
 '9757',
 '7729',
 '4578',
 '1414',
 '4028',
 '8473',
 '6855',
 '8113',
 '4369',
 '3934',
 '4189',
 '7187',
 '7373',
 '7003',
 '6332',
 '6016',
 '8354']

In [3]:
#def get_individual_stocks_df(
#self,
#specific_date: date = date.today(),
#columns_selected: list[str] = [],
#unique: bool = True
#) -> pl.DataFrame:
'''
specific_dateにおける最新ポートフォリオから、個別株のリストを取得する。
ETFは除外。
columns_selectedを指定すると、返すpl.DataFrameの列を選別(select)できる。指定しない場合は列の選別はしない。
uniqueを指定すると、返すpl.DataFrameにレコード重複があった場合、重複を排除する。
'''


# para
self = PM
specific_date = date.today()
columns_selected = ["ticker_code", "銘柄名"]
unique = True

In [5]:
PM.get_individual_stocks_df(columns_selected = columns_selected)

ticker_code,銘柄名
str,str
"""1301""","""極洋"""
"""1414""","""ショーボンド HD"""
"""1721""","""コムシス HD"""
"""2325""","""NJS"""
"""2393""","""日本ケアサプライ"""
…,…
"""8473""","""SBI HD"""
"""8591""","""オリックス"""
"""9433""","""KDDI"""


In [20]:
# body
df = self.df

# 指定日の最新portfolioを抽出
df = self.filter_portfolio_as_of_specific_date(specific_date)

# 個別株のみ選別
df = df.filter(
    pl.col("instrument_type") == "個別株"
)

# 列の選別
if len(columns_selected) != 0:
    df = df[columns_selected]

# recordの重複排除
if unique:
    df = df.unique()

# ticker_codeでsort
df = df.sort(["ticker_code"])


In [21]:
df

ticker_code,銘柄名
str,str
"""1301""","""極洋"""
"""1414""","""ショーボンド HD"""
"""1721""","""コムシス HD"""
"""2325""","""NJS"""
"""2393""","""日本ケアサプライ"""
…,…
"""8473""","""SBI HD"""
"""8591""","""オリックス"""
"""9433""","""KDDI"""


In [14]:
df.columns

['date',
 'ticker_code',
 'SecurityFirm_id',
 'purchase_price',
 'close_price',
 'quantity',
 'account_type',
 'description_x',
 'id',
 '銘柄名',
 'asset_class',
 'instrument_type',
 'is_domestic',
 'description',
 'SecurityFirm_name']

In [12]:
set(df["銘柄名"].to_list())

{'ENEOS HD',
 'KDDI',
 'KHネオケム',
 'NJS',
 'PILLAR',
 'SBI HD',
 'SHIFT',
 'ふくおか FG',
 'アイドマHD',
 'アクリート',
 'アセンテック',
 'オリックス',
 'キオクシアＨＤ',
 'コムシス HD',
 'ショーボンド HD',
 'ジェイリース',
 'ジャパンエンジンコーポレーション',
 'セルシス',
 'ソフトバンク',
 'ダイキン工業',
 'ツルハ HD',
 'デクセリアルズ',
 'デリカフーズ HD',
 'トリケミカル研究所',
 'フルヤ金属',
 'ベネフィットジャパン',
 'ボードルア',
 'ヤプリ',
 'ユニ・チャーム',
 'ユーザーローカル',
 'ラウンドワン',
 '三井E&S',
 '三井住友 FG',
 '三櫻工業',
 '三菱UFJ FG',
 '中国塗料',
 '丸紅',
 '久世',
 '伊藤忠商事',
 '住友ゴム工業',
 '大塚 HD',
 '小森コーポレーション',
 '日本たばこ産業',
 '日本アビオニクス',
 '日本ケアサプライ',
 '日本ヒューム',
 '日本製鉄',
 '日本電子材料',
 '日産化学',
 '明電舎',
 '月島 HD',
 '東京精密',
 '東京計器',
 '栗田工業',
 '極洋',
 '石原産業',
 '船井総研 HD',
 '豊和工業',
 '野村マイクロ・サイエンス',
 '鶴見製作所'}

In [3]:
#def filter_portfolio_as_of_specific_date(
#self,
#specific_date: date = date.today(),
#inplace: bool = False
#) -> pl.DataFrame | None:
'''
PortfolioManager.dfから、指定日における最新日のデータを抽出する。
(最も古いデータは2026年1月1日)
inplace = Falseの場合は、抽出結果をpl.DataFrameで返す。
inplace = Trueの場合は、抽出結果をPortfolioManager.dfにセットしてNoneを返す。
'''

# imoprt
from wequant.data_processing import DATA_DIR

# para
self = PM
specific_date = date(2026,1,23)
inplace = True


In [6]:
# test
PM.filter_portfolio_as_of_specific_date(specific_date=specific_date, inplace=inplace)
PM.df.select([
    "date",
    "ticker_code",
    "銘柄名",
    "asset_class",
    "instrument_type"
])

date,ticker_code,銘柄名,asset_class,instrument_type
date,str,str,str,str
2026-01-17,"""03311187""","""eMAXIS Slim 米国株式(S&P500)""","""株""","""投資信託"""
2026-01-17,"""1326""","""SPDRゴールド・トラスト""","""商品""","""ETF"""
2026-01-17,"""1414""","""ショーボンド HD""","""株""","""個別株"""
2026-01-17,"""1489""","""NEXT FUNDS日経平均高配当株50指数連動型上場投信""","""株""","""ETF"""
2026-01-17,"""1489""","""NEXT FUNDS日経平均高配当株50指数連動型上場投信""","""株""","""ETF"""
…,…,…,…,…
2026-01-17,"""N/A0001""","""楽天・マネーファンド""","""現金同等債券""","""投資信託"""
2026-01-17,"""SIL""","""グローバルＸ 銀ビジネス ETF""","""株""","""ETF"""
2026-01-17,"""円""","""円""","""現金""","""現金"""


In [18]:
# body
df = self.df

df = df.filter(
    pl.col("date") <= specefic_date
)
latest_date = df["date"].max()

df = df.filter(
    pl.col("date") == specefic_date
)

if inplace:
    self.df = df
    #return
else:
    return_=df



In [19]:
self.df

date,ticker_code,SecurityFirm_id,purchase_price,close_price,quantity,account_type,description_x,id,銘柄名,asset_class,instrument_type,is_domestic,description,SecurityFirm_name
date,str,str,"decimal[10,5]","decimal[10,5]","decimal[9,2]",str,str,str,str,str,str,str,str,str
2026-02-07,"""03311187""","""56551336704""",1.21000,3.90000,597873.00,"""特定口座""","""""","""56556965124""","""eMAXIS Slim 米国株式(S&P500)""","""株""","""投資信託""","""海外""","""""","""楽天証券"""
2026-02-07,"""1301""","""56551346048""",5180.00000,5180.00000,10.00,"""特定口座""","""""","""56551620864""","""極洋""","""株""","""個別株""","""国内""","""""","""SBI証券"""
2026-02-07,"""1326""","""56551336704""",52795.83000,69970.00000,6.00,"""特定口座""","""""","""56551621024""","""SPDRゴールド・トラスト""","""商品""","""ETF""","""海外""","""""","""楽天証券"""
2026-02-07,"""1414""","""56551346048""",1298.00000,1485.00000,40.00,"""特定口座""","""""","""56551621379""","""ショーボンド HD""","""株""","""個別株""","""国内""","""""","""SBI証券"""
2026-02-07,"""1489""","""56551346048""",2228.00000,3200.00000,646.00,"""特定口座""","""""","""56551621857""","""NEXT FUNDS日経平均高配当株50指数連動型上場投信""","""株""","""ETF""","""国内""","""""","""SBI証券"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-02-07,"""N/A0001""","""56551336704""",1.00000,1.00000,4511522.00,"""特定口座""","""""","""56556978756""","""楽天・マネーファンド""","""現金同等債券""","""投資信託""","""国内""","""""","""楽天証券"""
2026-02-07,"""SIL""","""56551336704""",7705.02000,15310.68000,14.00,"""特定口座""","""""","""56556984068""","""グローバルＸ 銀ビジネス ETF""","""株""","""ETF""","""海外""","""""","""楽天証券"""
2026-02-07,"""円""","""56551336704""",1.00000,1.00000,1987502.00,"""現金""","""""",null,"""円""","""現金""","""現金""","""国内""","""""","""楽天証券"""
